# 03. 합치기 + 학습/테스트 나누기 + 최종 채점 (v2)
**작성자: C**

v2에서 바뀐 점 (처음 틀 `03_merge_split_C`는 그대로 둠)
1. `USE_DUMMY` 스위치 추가 → 진짜 파일이 없어도 더미로 끝까지 실행해볼 수 있음 (더미일 땐 아무 파일도 저장 안 함)
2. 합치기 전 점검: 중복 센서, 통째로 빈 열, 한쪽 파일에만 있는 센서, 두 파일의 구·좌표가 같은지
3. 테스트 5개 구를 **서울 5개 권역(도심·동북·서북·서남·동남)에서 1개씩** 뽑음. 그냥 무작위로 뽑으면 seed 42에서 강동·동대문·성동·송파·중랑(전부 동쪽)이 나와서 바꿈
4. 최종 채점 코드 완성: `FINAL_RUN=True`일 때 **딱 한 번만** 돌고, 두 번째 실행은 막힘

- 6주차 1·2단계: 배포 상황 = '센서 없는 새 동네' → **자치구 단위**로 나누고 테스트는 잠금
- ⚠️ `test_LOCKED.csv`는 **공유 폴더가 아니라 C 개인 드라이브**에 저장

In [13]:
# ── 공통 설정 (모든 노트북 맨 위에 그대로 두기) ──
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AICEE_project'   # 모두 같은 경로 (공유 폴더를 '내 드라이브에 바로가기 추가' 해야 함)
DATA = f'{BASE}/data_processed'
RESULTS = f'{BASE}/results'

import sys
sys.path.append(f'{BASE}/shared')
RANDOM_STATE = 42

USE_DUMMY = False   # ← 진짜 데이터 나오면 False로 바꾸기

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os
import numpy as np, pandas as pd

FEATURES = ['green_300', 'impervious_300', 'water_300', 'dist_han_m']   # 정준 VIF 확인 후 팀 공통으로 수정
TARGETS = ['uhi_day', 'uhi_night']
TEST_PATH = '/content/drive/MyDrive/test_LOCKED.csv'   # C 개인 드라이브 (공유 폴더 밖)
assert not TEST_PATH.startswith(BASE), '테스트 파일이 공유 폴더 안에 저장되면 안 됨'

## 1) 불러오기

In [15]:
sfx = '_dummy' if USE_DUMMY else ''
labels = pd.read_csv(f'{DATA}/labels{sfx}.csv', dtype={'sensor_id': str})     # 예슬
feats = pd.read_csv(f'{DATA}/features{sfx}.csv', dtype={'sensor_id': str})    # 정준 (02_features_B_v2로 만든 파일)
print('labels  ', labels.shape)
print('features', feats.shape)

labels   (961, 6)
features (961, 8)


## 2) 합치기 전 점검
문제가 있으면 여기서 멈춤 (assert). 멈추면 고치지 말고 파일 주인에게 단톡으로 요청

In [16]:
assert labels['sensor_id'].is_unique, 'labels에 중복 센서 있음 → 예슬에게'
assert feats['sensor_id'].is_unique, 'features에 중복 센서 있음 → 정준에게'
assert set(TARGETS) <= set(labels.columns), f'labels 열 이름 확인: {labels.columns.tolist()}'
assert set(FEATURES) <= set(feats.columns), f'features 열 이름 확인: {feats.columns.tolist()}'

# 통째로 비어 있는 열이 있으면 dropna()에서 센서가 전부 사라짐
empty = [c for c in FEATURES + TARGETS if (labels if c in TARGETS else feats)[c].isna().all()]
assert not empty, f'값이 하나도 없는 열: {empty}'
print('빈칸 개수 (features):'); print(feats[FEATURES].isna().sum().to_string())
print('빈칸 개수 (labels):');   print(labels[TARGETS].isna().sum().to_string())

only_lab = sorted(set(labels['sensor_id']) - set(feats['sensor_id']))
only_feat = sorted(set(feats['sensor_id']) - set(labels['sensor_id']))
print(f'\nlabels에만 있는 센서 {len(only_lab)}개:', only_lab[:20])
print(f'features에만 있는 센서 {len(only_feat)}개:', only_feat[:20])

빈칸 개수 (features):
green_300         0
impervious_300    0
water_300         0
dist_han_m        0
빈칸 개수 (labels):
uhi_day      0
uhi_night    0

labels에만 있는 센서 0개: []
features에만 있는 센서 0개: []


두 파일의 구·좌표가 같은지 (합칠 땐 labels의 gu·lat·lon을 씀)

In [17]:
chk = labels[['sensor_id', 'gu', 'lat', 'lon']].merge(
    feats[['sensor_id', 'gu', 'lat', 'lon']], on='sensor_id', suffixes=('_A', '_B'))
gap_m = np.hypot((chk.lat_A - chk.lat_B) * 111_320,
                 (chk.lon_A - chk.lon_B) * 111_320 * np.cos(np.radians(chk.lat_A)))
print('구 이름이 다른 센서:', (chk.gu_A != chk.gu_B).sum())
print('좌표가 10m 넘게 다른 센서:', (gap_m > 10).sum())
chk[(chk.gu_A != chk.gu_B) | (gap_m > 10)].head(10)   # 0개가 정상. 있으면 예슬·정준 확인

구 이름이 다른 센서: 0
좌표가 10m 넘게 다른 센서: 0


,sensor_id,gu_A,lat_A,lon_A,gu_B,lat_B,lon_B


## 3) 합치기

In [18]:
data = labels.merge(feats.drop(columns=['gu', 'lat', 'lon']), on='sensor_id',
                    how='inner', validate='one_to_one')
n0 = len(data)
data = data.dropna(subset=FEATURES + TARGETS).reset_index(drop=True)
print(f'합친 센서 {n0}개 → 빈칸 있는 {n0 - len(data)}개 제외 → {len(data)}개, {data.gu.nunique()}개 구')

# 값 범위 점검 (비율은 0~1, 세 비율 합은 1 이하: 나지 등은 어디에도 안 들어가서 1보다 작을 수 있음)
ratio = ['green_300', 'impervious_300', 'water_300']
assert data[ratio].min().min() >= 0 and data[ratio].max().max() <= 1, '비율이 0~1 밖'
s = data[ratio].sum(axis=1)
print(f'세 비율 합: 최소 {s.min():.3f}, 중앙값 {s.median():.3f}, 최대 {s.max():.3f}')
print('구별 센서 수:'); print(data.groupby('gu').size().describe().round(1).to_string())

합친 센서 961개 → 빈칸 있는 0개 제외 → 961개, 25개 구
세 비율 합: 최소 0.873, 중앙값 0.999, 최대 1.000
구별 센서 수:
count    25.0
mean     38.4
std      17.1
min      22.0
25%      26.0
50%      33.0
75%      39.0
max      75.0


## 4) 테스트 5개 구 고르기: 권역별 1개 (6주차 2단계)
서울시 5대 생활권역에서 구를 하나씩 → 강남·강북, 한강변, 산자락이 자연스럽게 섞임.
권역 안에서는 `random_state=42`로 무작위. **결과를 보고 다시 뽑지 않기** (다시 뽑으면 이유를 기록)

In [19]:
REGION = {
    '도심권': ['종로구', '중구', '용산구'],
    '동북권': ['성동구', '광진구', '동대문구', '중랑구', '성북구', '강북구', '도봉구', '노원구'],
    '서북권': ['은평구', '서대문구', '마포구'],
    '서남권': ['양천구', '강서구', '구로구', '금천구', '영등포구', '동작구', '관악구'],
    '동남권': ['서초구', '강남구', '송파구', '강동구'],
}
gu_to_region = {g: r for r, gs in REGION.items() for g in gs}
unknown = sorted(set(data['gu']) - set(gu_to_region))
assert not unknown, f'권역표에 없는 구 이름: {unknown} (표기 확인)'
data['region'] = data['gu'].map(gu_to_region)

rng = np.random.default_rng(RANDOM_STATE)
present = set(data['gu'])
test_gu = [rng.choice(sorted(g for g in REGION[r] if g in present)) for r in REGION]
test_gu = [str(g) for g in test_gu]

summary = data.assign(set=np.where(data.gu.isin(test_gu), 'test', 'train')) \
              .groupby(['region', 'set']).size().unstack(fill_value=0)
print('테스트 구:', test_gu)
print(f"테스트 센서 {data.gu.isin(test_gu).sum()}개 / 전체 {len(data)}개 "
      f"({data.gu.isin(test_gu).mean():.0%})")
summary

테스트 구: ['용산구', '성북구', '서대문구', '금천구', '강동구']
테스트 센서 151개 / 전체 961개 (16%)


set,test,train
region,,
도심권,24,84
동남권,58,106
동북권,22,242
서남권,22,277
서북권,25,101


학습·테스트가 비슷한 동네로 구성됐는지 (테스트가 학습 범위 밖이면 트리 모델은 외삽 못 함 → 5·6주차)

In [20]:
import sys, importlib
p = f'{BASE}/shared'
if p not in sys.path:
    sys.path.append(p)
sys.path_importer_cache.pop(p, None)   # '폴더 없음' 기억 지우기
importlib.invalidate_caches()
import evaluate
print('불러오기 성공:', evaluate.__file__)

불러오기 성공: /content/drive/MyDrive/AICEE_project/shared/evaluate.py


In [21]:
from evaluate import outside_range
train = data[~data['gu'].isin(test_gu)].reset_index(drop=True)
test = data[data['gu'].isin(test_gu)].reset_index(drop=True)

compare = pd.concat({'train': train[FEATURES + TARGETS].describe().loc[['mean', 'min', 'max']],
                     'test': test[FEATURES + TARGETS].describe().loc[['mean', 'min', 'max']]})
print('테스트 센서 중 학습 범위 밖:', int(outside_range(test[FEATURES], train[FEATURES]).sum()), '개')
compare.round(3)

테스트 센서 중 학습 범위 밖: 0 개


green_300  impervious_300  water_300  dist_han_m  uhi_day  \
train mean      0.154           0.833      0.009    3974.511    0.022   
      min       0.000           0.009      0.000      43.000   -3.114   
      max       0.991           1.000      0.413   15382.000    2.978   
test  mean      0.156           0.834      0.004    3532.901   -0.034   
      min       0.000           0.331      0.000      45.000   -1.956   
      max       0.664           1.000      0.337    9313.000    2.038   

            uhi_night  
train mean     -0.061  
      min      -2.712  
      max       3.611  
test  mean     -0.353  
      min      -2.229  
      max       1.136

## 5) 저장 + 단톡 공지 (진짜 데이터일 때만 저장)

In [22]:
if USE_DUMMY:
    print('더미 데이터라 저장하지 않음 (train_dummy.csv는 팀원들이 쓰는 파일이라 덮어쓰면 안 됨)')
else:
    out = f'{DATA}/train.csv'
    assert not os.path.exists(out), 'train.csv가 이미 있음 → 덮어쓰지 말고 train_v2.csv로 저장하고 단톡 공지'
    assert not os.path.exists(TEST_PATH), 'test_LOCKED.csv가 이미 있음 → 분할을 다시 한 이유를 기록하고 v2로'
    train.drop(columns='region').to_csv(out, index=False)
    test.drop(columns='region').to_csv(TEST_PATH, index=False)
    print('저장 완료:', out, '/', TEST_PATH)

print(f'''
[파일 완성] train.csv
- 위치: data_processed/train.csv
- 내용: 센서 {len(train)}개, {train.gu.nunique()}개 구, 열: {', '.join(c for c in train.columns if c != 'region')}
- 테스트 구 (권역별 1개, 이 구들은 학습에 없음): {', '.join(test_gu)} / 테스트 센서 {len(test)}개는 채현 개인 드라이브에만
- 특이사항: labels {len(labels)}개 + features {len(feats)}개 → 합친 센서 {len(data)}개
- 다음 담당: 전원 (각자 모델 노트북에서 USE_DUMMY = False)
''')

AssertionError: train.csv가 이미 있음 → 덮어쓰지 말고 train_v2.csv로 저장하고 단톡 공지

## 6) 최종 채점: 모델 4개 확정 후 **딱 한 번** (6주차 6단계)
1. 각 모델을 train 전체로 학습 → test 예측
2. RMSE + **95% 신뢰구간** (부트스트랩 2,000번)
3. 모델 차이는 **paired bootstrap**: 구간이 0을 포함하면 '통계적으로 구분 안 됨'

- 각 모델의 설정·입력 변수는 담당자가 확정한 값으로 아래 `MODELS`를 채우기 (12/2까지 받기)
- 진짜 데이터에서는 `FINAL_RUN = True`로 바꿔야만 돌고, 한 번 돌면 `results/C_final_test_scores.csv`가 생겨서 두 번째는 막힘
- 더미일 때는 코드 확인용으로 아무 때나 돌려도 됨 (저장 안 함)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# (모델, 입력 변수) ← TODO: 12/2에 담당자에게 받은 최종 설정으로 교체
MODELS = {
    'baseline_mean': (DummyRegressor(strategy='mean'), ['green_300']),
    'model1_LR':     (make_pipeline(StandardScaler(), LinearRegression()), ['green_300']),
    'model2_MLR':    (make_pipeline(StandardScaler(), LinearRegression()), FEATURES),
    'model3_RF':     (RandomForestRegressor(n_estimators=500, min_samples_leaf=5,
                                            random_state=RANDOM_STATE, n_jobs=-1), FEATURES),
    'model4_XGB':    (XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.05, subsample=0.8,
                                   colsample_bytree=0.8, random_state=RANDOM_STATE), FEATURES),
}
FINAL_RUN = False   # ← 진짜 데이터 + 모델 4개 확정된 날 딱 한 번 True

In [ ]:
from sklearn.base import clone
from evaluate import metrics, bootstrap_ci, paired_bootstrap

LOCK = f'{RESULTS}/C_final_test_scores.csv'
if USE_DUMMY:
    tr, te = train, test   # 메모리 안의 더미 분할로 코드만 확인
else:
    assert FINAL_RUN, '최종 채점은 FINAL_RUN = True일 때만'
    assert not os.path.exists(LOCK), '이미 채점함 (테스트는 한 번만). 다시 하면 이유를 보고서에 기록'
    tr = pd.read_csv(f'{DATA}/train.csv')
    te = pd.read_csv(TEST_PATH)

rows, preds = [], {}
for t in TARGETS:
    for name, (est, cols) in MODELS.items():
        p = clone(est).fit(tr[cols], tr[t]).predict(te[cols])
        preds[(name, t)] = p
        rmse, (lo, hi) = bootstrap_ci(te[t], p)
        m = metrics(te[t], p)
        rows.append({'target': t, 'model': name, 'RMSE': rmse, 'CI_low': lo, 'CI_high': hi,
                     'MAE': m['MAE'], 'R2': m['R2'], 'n_test': len(te)})
scores = pd.DataFrame(rows).round(3)

# 모델 차이 (RMSE 차이 A - B의 95% 구간, 음수면 A가 더 좋음)
pairs = [('model1_LR', 'baseline_mean'), ('model2_MLR', 'model1_LR'),
         ('model3_RF', 'model2_MLR'), ('model4_XGB', 'model3_RF'), ('model4_XGB', 'baseline_mean')]
diff = []
for t in TARGETS:
    for a, b in pairs:
        lo, hi = paired_bootstrap(te[t], preds[(a, t)], preds[(b, t)])
        diff.append({'target': t, 'A': a, 'B': b, 'diff_low': lo, 'diff_high': hi,
                     '판정': 'A가 더 좋음' if hi < 0 else ('B가 더 좋음' if lo > 0 else '구분 안 됨')})
diff = pd.DataFrame(diff).round(3)

if not USE_DUMMY:
    scores.to_csv(LOCK, index=False)
    diff.to_csv(f'{RESULTS}/C_final_paired_bootstrap.csv', index=False)
    print('저장 완료 (이제 이 셀은 다시 안 돎):', LOCK)
display(scores) if 'display' in dir() else print(scores)
diff

## 7) 실패 사례 후보: 학습에 없던 구에서 크게 틀린 센서 (C 담당)
채점이 끝난 뒤 보는 분석이라 괜찮음 (이걸 보고 모델을 다시 고치면 안 됨).
오차 큰 센서를 지도·로드뷰로 확인해서 **왜 틀렸는지** 메모. 교차검증 쪽 후보는 13번 노트북 `worst_sensors`

In [ ]:
from evaluate import worst_sensors
best = scores[scores.target == 'uhi_night'].sort_values('RMSE').iloc[0]['model']
print('밤 RMSE 가장 낮은 모델:', best)
worst_sensors(te[['sensor_id', 'gu', 'lat', 'lon'] + FEATURES], te['uhi_night'], preds[(best, 'uhi_night')], k=10)

## 보고서용 메모 (결과 나오자마자 2~3문장씩)
- 데이터 합치기: labels __개 + features __개 → __개 센서, 제외 사유:
- 테스트 구 (권역별): / 테스트 센서 __개 (__%)
- 최종 결과 (RMSE, 95% CI):
- 실패 사례:
- 생성형 AI 사용 내역: Claude로 v2 작성 (더미 스위치, 합치기 점검, 권역별 분할, 최종 채점 코드). 확인한 방법: 더미 데이터로 끝까지 실행 / 직접 고친 부분: